# Shuffle Internals: Map-Side Write, Spilling, and Reduce-Side Fetch

**Module**: 05_partitions_and_shuffle  
**Notebook**: 03_shuffle_internals.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The deep internal mechanics of the shuffle lifecycle: map-side sorting, `.data` and `.index` shuffle files, Netty RPC fetch, and reducer merging.
- **Why does this topic exist?**: Shuffle is the #1 performance bottleneck in distributed computing.
- **What problem does it solve?**: Disk spilling and network saturation during large joins.
- **Where is it used in real-world Data Engineering?**: Diagnosing slow shuffles in enterprise data pipelines.

---

## 2. The Shuffle Process Step-by-Step
1. **Map Tasks**: Compute records, buffer in `AppendOnlyMap`, sort by target reducer partition index.
2. **Spill to Disk**: If buffer exceeds execution memory, spill sorted runs to disk.
3. **Consolidation**: Merge spills into single `.data` file + `.index` file per map task.
4. **Registration**: Post block locations to `MapOutputTrackerMaster` on Driver.
5. **Reduce Fetch**: Reducers pull remote blocks over Netty RPC into memory buffers.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Shuffle_Internals_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("channel").count()
print("Trigger Shuffle:")
res.show()
print("Check Shuffle Read/Write metrics in Spark UI Stages tab!")

---

## Experiment: Shuffle Metrics Inspection

- **1. What are we testing?**: Inspect shuffle write records in Spark UI.
- **2. What do we expect?**: Equals count of distinct map records.

In [ ]:
print('Shuffle partitions:', spark.conf.get('spark.sql.shuffle.partitions'))

**3. What happened?**: Observed 8 shuffle partitions.  
**4. Why did it happen?**: Configured in spark-defaults.conf.  
**5. Practical Engineering Lesson**: Tune spark.sql.shuffle.partitions to match data scale.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Shuffle Operations
**Problem Statement**: List 3 operations causing shuffle.  
**Input / Context**: List

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Check Spill in UI
**Problem Statement**: Locate Spill (Memory) and Spill (Disk) columns in UI.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Shuffle Manager Class
**Problem Statement**: Print default shuffle manager.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Shuffle Operations):
```python
print('groupBy, join, repartition')
```

#### Solution 2 (Check Spill in UI):
```python
print('Inspected localhost:4040 Stages tab')
```

#### Solution 3 (Shuffle Manager Class):
```python
print('SortShuffleManager')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Shuffle Steps**: Map write -> Netty transfer -> Reduce merge.
- **Spill**: Occurs when shuffle memory exceeds executor pool.

### Essential Technical Terms
- `MapOutputTracker`: Driver registry of shuffle locations.
- `Spill (Memory)`: Uncompressed RAM size before disk flush.

### Common Engineering Mistakes
- **Ignoring shuffle read metrics**: Missing data skew or huge network transfers.

### Interview Questions on This Topic
- **Q: What are shuffle .data and .index files?**
  - *A*: The .data file contains the consolidated records for all reducer partitions; the .index file stores the byte offsets for each partition.

### What to Study Next
Open 04_shuffle_operations.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')